# Ray Tracing Performance Analysis

## Overview
This notebook analyzes the performance of a ray tracing application across different configurations:
- **Rendering modes**: Single-Threaded, std::thread, and OpenMP
- **Thread counts**: 1, 2, 4, 8, 16
- **Ray counts**: 3,600, 10,800, 36,000, 108,000
- **Build modes**: Debug and Release

We'll examine timing, speedup, and parallel efficiency, with special focus on how Debug vs Release builds impact performance.

In [12]:
import polars as pl
import plotly.graph_objects as go
import plotly.express as px
from plotly.subplots import make_subplots


# Load the data
mac_df = pl.read_csv("../data/ray_tracing_performance_2026_03_03.csv")
pace_df = pl.read_csv("../data/pace_ray_tracing_performance.csv")

# Convert elapsed microseconds to milliseconds
mac_df = mac_df.with_columns(
    elapsed_ms=pl.col("elapsed microseconds") / 1000.0
).with_columns(environment=pl.lit("mac"))
pace_df = pace_df.with_columns(
    elapsed_ms=pl.col("elapsed microseconds") / 1000.0
).with_columns(environment=pl.lit("pace"))

df = pl.concat([mac_df, pace_df], how="vertical")

print(f"Data shape: {df.shape}")
print(f"Columns: {df.columns}")
print(f"\nFirst few rows:")
print(df.head(3))

Data shape: (11672, 9)
Columns: ['timestamp', 'render mode', 'thread count', 'ray count', 'elapsed microseconds', 'build mode', 'sampleCnt', 'elapsed_ms', 'environment']

First few rows:
shape: (3, 9)
┌─────────────┬─────────────┬────────┬───────┬───┬───────┬─────────────┬─────────────┬─────────────┐
│ timestamp   ┆ render mode ┆ thread ┆ ray   ┆ … ┆ build ┆ sampleCnt   ┆ elapsed_ms  ┆ environment │
│ ---         ┆ ---         ┆ count  ┆ count ┆   ┆ mode  ┆ ---         ┆ ---         ┆ ---         │
│ str         ┆ str         ┆ ---    ┆ ---   ┆   ┆ ---   ┆ i64         ┆ f64         ┆ str         │
│             ┆             ┆ i64    ┆ i64   ┆   ┆ str   ┆             ┆             ┆             │
╞═════════════╪═════════════╪════════╪═══════╪═══╪═══════╪═════════════╪═════════════╪═════════════╡
│ 2026-03-03  ┆ Single-Thre ┆ 1      ┆ 3600  ┆ … ┆ Debug ┆ 1           ┆ 4.618       ┆ mac         │
│ 16:23:07.77 ┆ aded        ┆        ┆       ┆   ┆       ┆             ┆             ┆      

In [13]:
avg_ms_by_render_ray_release = (
    df.filter(
        pl.col("build mode") == "Release", pl.col("render mode") != "Single-Threaded"
    )
    .group_by("ray count", "thread count", "render mode", "environment")
    .agg(avg_elapsed_ms=pl.col("elapsed_ms").mean())
    .pivot(
        on="render mode",
        index=["ray count", "thread count", "environment"],
        values="avg_elapsed_ms",
        sort_columns=True,
    )
    .sort(["ray count", "thread count", "environment"])
)

avg_ms_by_render_ray_debug = (
    df.filter(
        pl.col("build mode") == "Debug", pl.col("render mode") != "Single-Threaded"
    )
    .group_by("ray count", "thread count", "render mode", "environment")
    .agg(avg_elapsed_ms=pl.col("elapsed_ms").mean())
    .pivot(
        on="render mode",
        index=["ray count", "thread count", "environment"],
        values="avg_elapsed_ms",
        sort_columns=True,
    )
    .sort(["ray count", "thread count", "environment"])
)

avg_ms_by_single_threaded_debug = (
    df.filter(
        pl.col("build mode") == "Debug", pl.col("render mode") == "Single-Threaded"
    )
    .group_by("ray count", "thread count", "render mode", "environment")
    .agg(avg_elapsed_ms=pl.col("elapsed_ms").mean())
    .pivot(
        on="render mode",
        index=["ray count", "thread count", "environment"],
        values="avg_elapsed_ms",
        sort_columns=True,
    )
    .sort(["ray count", "thread count", "environment"])
)

avg_ms_by_single_threaded_release = (
    df.filter(
        pl.col("build mode") == "Release", pl.col("render mode") == "Single-Threaded"
    )
    .group_by("ray count", "thread count", "render mode", "environment")
    .agg(avg_elapsed_ms=pl.col("elapsed_ms").mean())
    .pivot(
        on="render mode",
        index=["ray count", "thread count", "environment"],
        values="avg_elapsed_ms",
        sort_columns=True,
    )
    .sort(["ray count", "thread count", "environment"])
)

In [14]:
single_threaded_data = avg_ms_by_single_threaded_release.select(["ray count", "Single-Threaded", "environment"])

avg_ms_by_render_ray_release = (
    avg_ms_by_render_ray_release.join(
        single_threaded_data, on=["ray count", "environment"], how="left"
    )
    .with_columns(
        OpenMP_SpeedUp=pl.col("Single-Threaded") / pl.col("OpenMP"),
        StdThread_SpeedUp=pl.col("Single-Threaded") / pl.col("StdThread"),
    )
    .sort("ray count", "thread count")
)

print(avg_ms_by_render_ray_release)

shape: (36, 8)
┌───────────┬────────┬─────────────┬──────────┬───────────┬─────────────┬─────────────┬────────────┐
│ ray count ┆ thread ┆ environment ┆ OpenMP   ┆ StdThread ┆ Single-Thre ┆ OpenMP_Spee ┆ StdThread_ │
│ ---       ┆ count  ┆ ---         ┆ ---      ┆ ---       ┆ aded        ┆ dUp         ┆ SpeedUp    │
│ i64       ┆ ---    ┆ str         ┆ f64      ┆ f64       ┆ ---         ┆ ---         ┆ ---        │
│           ┆ i64    ┆             ┆          ┆           ┆ f64         ┆ f64         ┆ f64        │
╞═══════════╪════════╪═════════════╪══════════╪═══════════╪═════════════╪═════════════╪════════════╡
│ 3600      ┆ 2      ┆ mac         ┆ 0.350772 ┆ 0.58704   ┆ 0.690267    ┆ 1.96785     ┆ 1.175845   │
│ 3600      ┆ 2      ┆ pace        ┆ 0.76081  ┆ 0.52696   ┆ 0.95608     ┆ 1.256661    ┆ 1.814331   │
│ 3600      ┆ 4      ┆ mac         ┆ 0.332416 ┆ 0.503317  ┆ 0.690267    ┆ 2.076518    ┆ 1.371437   │
│ 3600      ┆ 4      ┆ pace        ┆ 0.93331  ┆ 0.75261   ┆ 0.95608     ┆ 1.

In [15]:
single_threaded_data = avg_ms_by_single_threaded_debug.select(["ray count", "Single-Threaded", "environment"])

avg_ms_by_render_ray_debug = (
    avg_ms_by_render_ray_debug.join(
        single_threaded_data, on=["ray count", "environment"], how="left"
    )
    .with_columns(
        OpenMP_SpeedUp=pl.col("Single-Threaded") / pl.col("OpenMP"),
        StdThread_SpeedUp=pl.col("Single-Threaded") / pl.col("StdThread"),
    )
    .sort(["ray count", "thread count", "environment"])
)

print(avg_ms_by_render_ray_debug)

shape: (16, 8)
┌───────────┬────────┬─────────────┬───────────┬───────────┬─────────────┬────────────┬────────────┐
│ ray count ┆ thread ┆ environment ┆ OpenMP    ┆ StdThread ┆ Single-Thre ┆ OpenMP_Spe ┆ StdThread_ │
│ ---       ┆ count  ┆ ---         ┆ ---       ┆ ---       ┆ aded        ┆ edUp       ┆ SpeedUp    │
│ i64       ┆ ---    ┆ str         ┆ f64       ┆ f64       ┆ ---         ┆ ---        ┆ ---        │
│           ┆ i64    ┆             ┆           ┆           ┆ f64         ┆ f64        ┆ f64        │
╞═══════════╪════════╪═════════════╪═══════════╪═══════════╪═════════════╪════════════╪════════════╡
│ 3600      ┆ 2      ┆ mac         ┆ 3.005911  ┆ 3.369287  ┆ 4.423772    ┆ 1.471691   ┆ 1.31297    │
│ 3600      ┆ 4      ┆ mac         ┆ 2.708396  ┆ 2.741119  ┆ 4.423772    ┆ 1.633355   ┆ 1.613856   │
│ 3600      ┆ 8      ┆ mac         ┆ 2.214406  ┆ 2.892287  ┆ 4.423772    ┆ 1.997724   ┆ 1.529507   │
│ 3600      ┆ 16     ┆ mac         ┆ 2.722347  ┆ 2.509762  ┆ 4.423772    ┆ 1

In [16]:
avg_ms_by_render_ray_debug.write_csv("../data/avg_ms_by_render_ray_debug.csv")
avg_ms_by_render_ray_release.write_csv("../data/avg_ms_by_render_ray_release.csv")

In [ ]:
from plotly.subplots import make_subplots
import plotly.graph_objects as go

ray_counts = sorted(avg_ms_by_render_ray_release["ray count"].unique().to_list())

fig = make_subplots(
    rows=2,
    cols=2,
    subplot_titles=[f"Ray Count: {rc}" for rc in ray_counts],
    specs=[[{}, {}], [{}, {}]],
)

row_col_pairs = [(1, 1), (1, 2), (2, 1), (2, 2)]

for idx, ray_count in enumerate(ray_counts):
    row, col = row_col_pairs[idx]

    data_subset = avg_ms_by_render_ray_release.filter(
        pl.col("ray count") == ray_count
    ).sort("thread count")

    # OpenMP line
    fig.add_trace(
        go.Scatter(
            x=data_subset["thread count"].to_list(),
            y=data_subset["OpenMP"].to_list(),
            mode="lines+markers",
            name="OpenMP",
            line=dict(color="blue", width=2),
            showlegend=(idx == 0),
        ),
        row=row,
        col=col,
    )

    # StdThread line
    fig.add_trace(
        go.Scatter(
            x=data_subset["thread count"].to_list(),
            y=data_subset["StdThread"].to_list(),
            mode="lines+markers",
            name="StdThread",
            line=dict(color="orange", width=2),
            showlegend=(idx == 0),
        ),
        row=row,
        col=col,
    )

    fig.update_xaxes(title_text="Thread Count", row=row, col=col)
    fig.update_yaxes(title_text="Execution Time (ms)", row=row, col=col)

fig.update_layout(
    title_text="OpenMP vs StdThread - Release Execution Time Comparison",
    height=800,
    hovermode="closest",
)

fig.write_image("release_execution_time_comparison.svg")
fig.show()

In [ ]:
fig = make_subplots(
    rows=2,
    cols=2,
    subplot_titles=[f"Ray Count: {rc}" for rc in ray_counts],
    specs=[[{}, {}], [{}, {}]],
)

for idx, ray_count in enumerate(ray_counts):
    row, col = row_col_pairs[idx]

    data_subset = avg_ms_by_render_ray_release.filter(
        pl.col("ray count") == ray_count
    ).sort("thread count")

    # OpenMP SpeedUp
    fig.add_trace(
        go.Scatter(
            x=data_subset["thread count"].to_list(),
            y=data_subset["OpenMP_SpeedUp"].to_list(),
            mode="lines+markers",
            name="OpenMP",
            line=dict(color="blue", width=2),
            showlegend=(idx == 0),
        ),
        row=row,
        col=col,
    )

    # StdThread SpeedUp
    fig.add_trace(
        go.Scatter(
            x=data_subset["thread count"].to_list(),
            y=data_subset["StdThread_SpeedUp"].to_list(),
            mode="lines+markers",
            name="StdThread",
            line=dict(color="orange", width=2),
            showlegend=(idx == 0),
        ),
        row=row,
        col=col,
    )

    fig.update_xaxes(title_text="Thread Count", row=row, col=col)
    fig.update_yaxes(title_text="Speedup", row=row, col=col)

fig.update_layout(
    title_text="OpenMP vs StdThread - Speedup Comparison",
    height=800,
    hovermode="closest",
)

fig.write_image("release_speedup_comparison.svg")
fig.show()

In [ ]:
fig = make_subplots(
    rows=1, cols=2, subplot_titles=("OpenMP", "StdThread"), specs=[[{}, {}]]
)

for ray_count in ray_counts:
    data_subset = avg_ms_by_render_ray_release.filter(
        pl.col("ray count") == ray_count
    ).sort("thread count")

    # OpenMP
    fig.add_trace(
        go.Scatter(
            x=data_subset["thread count"].to_list(),
            y=data_subset["OpenMP"].to_list(),
            mode="lines+markers",
            name=f"Ray: {ray_count}",
            showlegend=True,
        ),
        row=1,
        col=1,
    )

    # StdThread
    fig.add_trace(
        go.Scatter(
            x=data_subset["thread count"].to_list(),
            y=data_subset["StdThread"].to_list(),
            mode="lines+markers",
            name=f"Ray: {ray_count}",
            showlegend=False,
        ),
        row=1,
        col=2,
    )

fig.update_xaxes(title_text="Thread Count", row=1, col=1)
fig.update_xaxes(title_text="Thread Count", row=1, col=2)
fig.update_yaxes(title_text="Execution Time (ms)", row=1, col=1)
fig.update_yaxes(title_text="Execution Time (ms)", row=1, col=2)

fig.update_layout(
    title_text="OpenMP vs StdThread by Ray Count", height=500, hovermode="closest"
)

fig.write_image("OpenMP_vs_StdThread_by_Ray_Count.svg")
fig.show()

In [ ]:
# Create charts for each ray count showing Debug vs Release for all render modes (including Single-Threaded)
fig = make_subplots(
    rows=2,
    cols=2,
    subplot_titles=[f"Ray Count: {rc}" for rc in ray_counts],
    specs=[[{}, {}], [{}, {}]],
)

for idx, ray_count in enumerate(ray_counts):
    row, col = row_col_pairs[idx]

    # Get data for this ray count from both builds
    release_data = avg_ms_by_render_ray_release.filter(
        pl.col("ray count") == ray_count
    ).sort("thread count")

    debug_data = avg_ms_by_render_ray_debug.filter(
        pl.col("ray count") == ray_count
    ).sort("thread count")

    # OpenMP Release
    fig.add_trace(
        go.Scatter(
            x=release_data["thread count"].to_list(),
            y=release_data["OpenMP"].to_list(),
            mode="lines+markers",
            name="OpenMP (Release)",
            line=dict(color="blue", width=2),
            marker=dict(size=6),
            showlegend=(idx == 0),
        ),
        row=row,
        col=col,
    )

    # OpenMP Debug
    fig.add_trace(
        go.Scatter(
            x=debug_data["thread count"].to_list(),
            y=debug_data["OpenMP"].to_list(),
            mode="lines+markers",
            name="OpenMP (Debug)",
            line=dict(color="blue", width=2, dash="dash"),
            marker=dict(size=6),
            showlegend=(idx == 0),
        ),
        row=row,
        col=col,
    )

    # StdThread Release
    fig.add_trace(
        go.Scatter(
            x=release_data["thread count"].to_list(),
            y=release_data["StdThread"].to_list(),
            mode="lines+markers",
            name="StdThread (Release)",
            line=dict(color="orange", width=2),
            marker=dict(size=6),
            showlegend=(idx == 0),
        ),
        row=row,
        col=col,
    )

    # StdThread Debug
    fig.add_trace(
        go.Scatter(
            x=debug_data["thread count"].to_list(),
            y=debug_data["StdThread"].to_list(),
            mode="lines+markers",
            name="StdThread (Debug)",
            line=dict(color="orange", width=2, dash="dash"),
            marker=dict(size=6),
            showlegend=(idx == 0),
        ),
        row=row,
        col=col,
    )

    # Single-Threaded Release
    fig.add_trace(
        go.Scatter(
            x=release_data["thread count"].to_list(),
            y=release_data["Single-Threaded"].to_list(),
            mode="lines+markers",
            name="Single-Threaded (Release)",
            line=dict(color="green", width=2),
            marker=dict(size=6),
            showlegend=(idx == 0),
        ),
        row=row,
        col=col,
    )

    # Single-Threaded Debug
    fig.add_trace(
        go.Scatter(
            x=debug_data["thread count"].to_list(),
            y=debug_data["Single-Threaded"].to_list(),
            mode="lines+markers",
            name="Single-Threaded (Debug)",
            line=dict(color="green", width=2, dash="dash"),
            marker=dict(size=6),
            showlegend=(idx == 0),
        ),
        row=row,
        col=col,
    )

    fig.update_xaxes(title_text="Thread Count", row=row, col=col)
    fig.update_yaxes(title_text="Execution Time (ms)", row=row, col=col)

fig.update_layout(
    title_text="Debug vs Release Build - Execution Time Comparison (Including Single-Threaded)",
    height=900,
    hovermode="closest",
    legend=dict(x=1.05, y=1),
)

fig.write_image("debug_vs_release_comparison.svg")
fig.show()

In [ ]:
# Print comparison table showing improvement from Debug to Release
print("=" * 120)
print("BUILD MODE IMPACT - Improvement Factor (Debug Time / Release Time)")
print("=" * 120)

for ray_count in ray_counts:
    print(f"\nRay Count: {ray_count}")
    print("-" * 80)

    release = avg_ms_by_render_ray_release.filter(
        pl.col("ray count") == ray_count
    ).sort("thread count")
    debug = avg_ms_by_render_ray_debug.filter(pl.col("ray count") == ray_count).sort(
        "thread count"
    )

    comparison = release.with_columns(
        OpenMP_Improvement=debug["OpenMP"] / release["OpenMP"],
        StdThread_Improvement=debug["StdThread"] / release["StdThread"],
    )

    print(
        f"{'Thread Count':<12} {'OpenMP Improvement':<20} {'StdThread Improvement':<20}"
    )
    print("-" * 80)

    for row in comparison.to_dicts():
        tc = row["thread count"]
        omp_imp = row["OpenMP_Improvement"]
        std_imp = row["StdThread_Improvement"]
        print(f"{tc:<12} {omp_imp:<20.2f}x {std_imp:<20.2f}x")

In [ ]:
# Create charts for each ray count showing Debug vs Release for both render modes
fig = make_subplots(
    rows=2,
    cols=2,
    subplot_titles=[f"Ray Count: {rc}" for rc in ray_counts],
    specs=[[{}, {}], [{}, {}]],
)

row_col_pairs = [(1, 1), (1, 2), (2, 1), (2, 2)]

for idx, ray_count in enumerate(ray_counts):
    row, col = row_col_pairs[idx]

    # Get data for this ray count from both builds
    release_data = avg_ms_by_render_ray_release.filter(
        pl.col("ray count") == ray_count
    ).sort("thread count")

    debug_data = avg_ms_by_render_ray_debug.filter(
        pl.col("ray count") == ray_count
    ).sort("thread count")

    # OpenMP Release
    fig.add_trace(
        go.Scatter(
            x=release_data["thread count"].to_list(),
            y=release_data["OpenMP"].to_list(),
            mode="lines+markers",
            name="OpenMP (Release)",
            line=dict(color="blue", width=2),
            showlegend=(idx == 0),
        ),
        row=row,
        col=col,
    )

    # OpenMP Debug
    fig.add_trace(
        go.Scatter(
            x=debug_data["thread count"].to_list(),
            y=debug_data["OpenMP"].to_list(),
            mode="lines+markers",
            name="OpenMP (Debug)",
            line=dict(color="blue", width=2, dash="dash"),
            showlegend=(idx == 0),
        ),
        row=row,
        col=col,
    )

    # StdThread Release
    fig.add_trace(
        go.Scatter(
            x=release_data["thread count"].to_list(),
            y=release_data["StdThread"].to_list(),
            mode="lines+markers",
            name="StdThread (Release)",
            line=dict(color="orange", width=2),
            showlegend=(idx == 0),
        ),
        row=row,
        col=col,
    )

    # StdThread Debug
    fig.add_trace(
        go.Scatter(
            x=debug_data["thread count"].to_list(),
            y=debug_data["StdThread"].to_list(),
            mode="lines+markers",
            name="StdThread (Debug)",
            line=dict(color="orange", width=2, dash="dash"),
            showlegend=(idx == 0),
        ),
        row=row,
        col=col,
    )

    fig.update_xaxes(title_text="Thread Count", row=row, col=col)
    fig.update_yaxes(title_text="Execution Time (ms)", row=row, col=col)

fig.update_layout(
    title_text="Debug vs Release Build - Execution Time Comparison by Ray Count",
    height=800,
    hovermode="closest",
)

fig.write_image("debug_vs_release_comparison.svg")
fig.show()

## 6. Debug vs Release Build Comparison by Ray Count

This section compares execution times across build modes for each ray count and render mode.

## 5. Summary and Key Findings

### Key Observations:

#### Speedup Characteristics
- **Higher ray counts show better speedup**: When there's more computational work (108,000 rays), parallelization is more effective
- **Diminishing returns at higher thread counts**: Both OpenMP and StdThread show diminishing speedup gains beyond 8 threads
- **OpenMP generally outperforms std::thread**: OpenMP achieves better speedup, particularly at higher ray counts

#### Parallel Efficiency
- **Efficiency decreases with more threads**: This is expected due to:
  - Thread synchronization overhead
  - Cache contention and false sharing
  - Load imbalance across threads
- **Low ray counts show poor efficiency**: 3,600 rays with 16 threads results in insufficient work per thread
- **Higher ray counts maintain better efficiency**: At 108,000 rays, efficiency stays more stable even with 16 threads

#### Build Mode Impact
- **Release build is critical for parallelization benefits**: The compiler optimizations enable better parallel performance
- **Debug builds show different efficiency patterns**: Overhead is more pronounced, making parallelization benefits smaller

#### Implementation Comparison
- **OpenMP is the better choice**: Superior speedup and efficiency across all configurations
- **std::thread shows more overhead**: Particularly noticeable at lower ray counts and higher thread counts

In [ ]:
# Calculate efficiency for Debug build
debug_efficiency = avg_ms_by_render_ray_debug.with_columns(
    OpenMP_Efficiency=pl.col("OpenMP_SpeedUp") / pl.col("thread count"),
    StdThread_Efficiency=pl.col("StdThread_SpeedUp") / pl.col("thread count"),
)

print("=" * 100)
print("DEBUG BUILD - EFFICIENCY ANALYSIS")
print("=" * 100)
print(
    debug_efficiency.select(
        ["ray count", "thread count", "OpenMP_Efficiency", "StdThread_Efficiency"]
    )
)

# Efficiency Plots - Debug Build
fig = make_subplots(
    rows=2,
    cols=2,
    subplot_titles=[f"Ray Count: {rc}" for rc in ray_counts],
    specs=[[{}, {}], [{}, {}]],
)

for idx, ray_count in enumerate(ray_counts):
    row, col = row_col_pairs[idx]

    data_subset = debug_efficiency.filter(pl.col("ray count") == ray_count).sort(
        "thread count"
    )

    # Perfect efficiency line (1.0)
    thread_counts = [1, 2, 4, 8, 16]
    fig.add_trace(
        go.Scatter(
            x=thread_counts,
            y=[1.0] * len(thread_counts),
            mode="lines",
            name="Perfect Efficiency",
            line=dict(color="black", dash="dash"),
            showlegend=(idx == 0),
        ),
        row=row,
        col=col,
    )

    # OpenMP efficiency
    fig.add_trace(
        go.Scatter(
            x=data_subset["thread count"].to_list(),
            y=data_subset["OpenMP_Efficiency"].to_list(),
            mode="lines+markers",
            name="OpenMP",
            line=dict(color="blue", width=2),
            showlegend=(idx == 0),
        ),
        row=row,
        col=col,
    )

    # StdThread efficiency
    fig.add_trace(
        go.Scatter(
            x=data_subset["thread count"].to_list(),
            y=data_subset["StdThread_Efficiency"].to_list(),
            mode="lines+markers",
            name="StdThread",
            line=dict(color="orange", width=2),
            showlegend=(idx == 0),
        ),
        row=row,
        col=col,
    )

    fig.update_xaxes(title_text="Thread Count", row=row, col=col)
    fig.update_yaxes(
        title_text="Efficiency (Speedup / N_threads)", row=row, col=col, range=[0, 1.2]
    )

fig.update_layout(
    title_text="Debug Build - Parallel Efficiency Analysis",
    height=800,
    hovermode="closest",
)

# fig.write_image('docs/efficiency_analysis_debug.svg')
fig.show()

In [ ]:
# Calculate efficiency for Debug build
release_efficiency = avg_ms_by_render_ray_release.with_columns(
    OpenMP_Efficiency=pl.col("OpenMP_SpeedUp") / pl.col("thread count"),
    StdThread_Efficiency=pl.col("StdThread_SpeedUp") / pl.col("thread count"),
)

# Efficiency Plots - Release Build
fig = make_subplots(
    rows=2,
    cols=2,
    subplot_titles=[f"Ray Count: {rc}" for rc in ray_counts],
    specs=[[{}, {}], [{}, {}]],
)

for idx, ray_count in enumerate(ray_counts):
    row, col = row_col_pairs[idx]

    data_subset = release_efficiency.filter(pl.col("ray count") == ray_count).sort(
        "thread count"
    )

    # Perfect efficiency line (1.0)
    thread_counts = [1, 2, 4, 8, 16]
    fig.add_trace(
        go.Scatter(
            x=thread_counts,
            y=[1.0] * len(thread_counts),
            mode="lines",
            name="Perfect Efficiency",
            line=dict(color="black", dash="dash"),
            showlegend=(idx == 0),
        ),
        row=row,
        col=col,
    )

    # OpenMP efficiency
    fig.add_trace(
        go.Scatter(
            x=data_subset["thread count"].to_list(),
            y=data_subset["OpenMP_Efficiency"].to_list(),
            mode="lines+markers",
            name="OpenMP",
            line=dict(color="blue", width=2),
            showlegend=(idx == 0),
        ),
        row=row,
        col=col,
    )

    # StdThread efficiency
    fig.add_trace(
        go.Scatter(
            x=data_subset["thread count"].to_list(),
            y=data_subset["StdThread_Efficiency"].to_list(),
            mode="lines+markers",
            name="StdThread",
            line=dict(color="orange", width=2),
            showlegend=(idx == 0),
        ),
        row=row,
        col=col,
    )

    fig.update_xaxes(title_text="Thread Count", row=row, col=col)
    fig.update_yaxes(
        title_text="Efficiency (Speedup / N_threads)", row=row, col=col, range=[0, 1.2]
    )

fig.update_layout(
    title_text="Release Build - Parallel Efficiency Analysis",
    height=800,
    hovermode="closest",
)

fig.write_image("efficiency_analysis_release.svg")
fig.show()

In [ ]:
# Calculate efficiency for Release build
release_efficiency = avg_ms_by_render_ray_release.with_columns(
    OpenMP_Efficiency=pl.col("OpenMP_SpeedUp") / pl.col("thread count"),
    StdThread_Efficiency=pl.col("StdThread_SpeedUp") / pl.col("thread count"),
)

print("=" * 100)
print("RELEASE BUILD - EFFICIENCY ANALYSIS")
print("=" * 100)
print(
    release_efficiency.select(
        ["ray count", "thread count", "OpenMP_Efficiency", "StdThread_Efficiency"]
    )
)

print("\n" + "=" * 100)
print("KEY OBSERVATIONS:")
print("=" * 100)
for ray_count in ray_counts:
    data = release_efficiency.filter(pl.col("ray count") == ray_count)
    omp_avg = data["OpenMP_Efficiency"].mean()
    std_avg = data["StdThread_Efficiency"].mean()
    print(
        f"Ray Count {ray_count:6d}: OpenMP avg efficiency = {omp_avg:.3f}, StdThread avg efficiency = {std_avg:.3f}"
    )

## 4. Parallel Efficiency Analysis

### Discussion: Parallel Efficiency

Parallel efficiency measures how well we utilize the available threads:

**Efficiency = Speedup / N_threads**

Where N_threads is the number of threads. Ideal efficiency is 1.0 (100%), meaning perfect scaling. Efficiency typically decreases as we add more threads due to:

1. **Thread Management Overhead**: Creating and managing more threads consumes resources
2. **Synchronization Costs**: More threads require more synchronization
3. **Cache Contention**: Multiple threads competing for cache lines can cause performance degradation
4. **Diminishing Returns**: After a certain point, adding more threads provides less benefit relative to the overhead

For this ray tracing workload, we expect efficiency to drop off at higher thread counts due to these factors.

In [ ]:
# Speedup Analysis - Debug Build
fig = make_subplots(
    rows=2,
    cols=2,
    subplot_titles=[f"Ray Count: {rc}" for rc in ray_counts],
    specs=[[{}, {}], [{}, {}]],
)

for idx, ray_count in enumerate(ray_counts):
    row, col = row_col_pairs[idx]

    data_subset = avg_ms_by_render_ray_debug.filter(
        pl.col("ray count") == ray_count
    ).sort("thread count")

    # Ideal linear speedup line
    thread_counts = [1, 2, 4, 8, 16]
    fig.add_trace(
        go.Scatter(
            x=thread_counts,
            y=thread_counts,
            mode="lines",
            name="Ideal Linear",
            line=dict(color="black", dash="dash"),
            showlegend=(idx == 0),
        ),
        row=row,
        col=col,
    )

    # OpenMP speedup
    fig.add_trace(
        go.Scatter(
            x=data_subset["thread count"].to_list(),
            y=data_subset["OpenMP_SpeedUp"].to_list(),
            mode="lines+markers",
            name="OpenMP",
            line=dict(color="blue", width=2),
            showlegend=(idx == 0),
        ),
        row=row,
        col=col,
    )

    # StdThread speedup
    fig.add_trace(
        go.Scatter(
            x=data_subset["thread count"].to_list(),
            y=data_subset["StdThread_SpeedUp"].to_list(),
            mode="lines+markers",
            name="StdThread",
            line=dict(color="orange", width=2),
            showlegend=(idx == 0),
        ),
        row=row,
        col=col,
    )

    fig.update_xaxes(title_text="Thread Count", row=row, col=col)
    fig.update_yaxes(title_text="Speedup (T₁/Tₙ)", row=row, col=col)

fig.update_layout(
    title_text="Debug Build - Speedup Analysis (vs Single-Threaded)",
    height=800,
    hovermode="closest",
)

fig.write_image("docs/speedup_analysis_debug.svg")
fig.show()

In [ ]:
# Speedup Analysis - Release Build
ray_counts = sorted(avg_ms_by_render_ray_release["ray count"].unique().to_list())

fig = make_subplots(
    rows=2,
    cols=2,
    subplot_titles=[f"Ray Count: {rc}" for rc in ray_counts],
    specs=[[{}, {}], [{}, {}]],
)

row_col_pairs = [(1, 1), (1, 2), (2, 1), (2, 2)]

for idx, ray_count in enumerate(ray_counts):
    row, col = row_col_pairs[idx]

    data_subset = avg_ms_by_render_ray_release.filter(
        pl.col("ray count") == ray_count
    ).sort("thread count")

    # Ideal linear speedup line
    thread_counts = [1, 2, 4, 8, 16]
    fig.add_trace(
        go.Scatter(
            x=thread_counts,
            y=thread_counts,
            mode="lines",
            name="Ideal Linear",
            line=dict(color="black", dash="dash"),
            showlegend=(idx == 0),
        ),
        row=row,
        col=col,
    )

    # OpenMP speedup
    fig.add_trace(
        go.Scatter(
            x=data_subset["thread count"].to_list(),
            y=data_subset["OpenMP_SpeedUp"].to_list(),
            mode="lines+markers",
            name="OpenMP",
            line=dict(color="blue", width=2),
            showlegend=(idx == 0),
        ),
        row=row,
        col=col,
    )

    # StdThread speedup
    fig.add_trace(
        go.Scatter(
            x=data_subset["thread count"].to_list(),
            y=data_subset["StdThread_SpeedUp"].to_list(),
            mode="lines+markers",
            name="StdThread",
            line=dict(color="orange", width=2),
            showlegend=(idx == 0),
        ),
        row=row,
        col=col,
    )

    fig.update_xaxes(title_text="Thread Count", row=row, col=col)
    fig.update_yaxes(title_text="Speedup (T₁/Tₙ)", row=row, col=col)

fig.update_layout(
    title_text="Release Build - Speedup Analysis (vs Single-Threaded)",
    height=800,
    hovermode="closest",
)

fig.write_image("docs/speedup_analysis_release.svg")
fig.show()

## 3. Speedup Analysis

### Discussion: Speedup (T₁/Tₙ) Analysis

Speedup measures how much faster a multi-threaded implementation is compared to single-threaded execution. For N threads, speedup is calculated as:

**Speedup(N) = T₁ / Tₙ**

Where:
- T₁ = execution time with 1 thread (single-threaded)
- Tₙ = execution time with N threads

Ideal linear speedup would be N (perfect parallelization), but in practice we rarely achieve this due to:
- Overhead of thread creation and management
- Synchronization costs
- Cache contention and false sharing
- Uneven work distribution

# Print timing tables
print("=" * 100)
print("RELEASE BUILD - EXECUTION TIME COMPARISON")
print("=" * 100)
print(avg_ms_by_render_ray_release.select(['ray count', 'thread count', 'OpenMP', 'StdThread', 'Single-Threaded']))

print("\n" + "=" * 100)
print("DEBUG BUILD - EXECUTION TIME COMPARISON")
print("=" * 100)
print(avg_ms_by_render_ray_debug.select(['ray count', 'thread count', 'OpenMP', 'StdThread', 'Single-Threaded']))

## 3. Speedup Analysis

### Discussion: Speedup (T₁/Tₙ) Analysis

Speedup measures how much faster a multi-threaded implementation is compared to single-threaded execution. For N threads, speedup is calculated as:

**Speedup(N) = T₁ / Tₙ**

Where:
- T₁ = execution time with 1 thread
- Tₙ = execution time with N threads

Ideal linear speedup would be N (perfect parallelization), but in practice we rarely achieve this due to:
- Overhead of thread creation and management
- Synchronization costs
- Cache contention and false sharing
- Uneven work distribution

### Speedup Plots: std::thread

These plots show speedup vs. number of threads for each ray count. The diagonal lines represent ideal linear speedup.

### Speedup Plots: OpenMP

## 4. Parallel Efficiency Analysis

### Discussion: Parallel Efficiency

Parallel efficiency measures how well we utilize the available threads:

**Efficiency = Speedup / N_threads**

Where N_threads is the number of threads. Ideal efficiency is 1.0 (100%), meaning perfect scaling. Efficiency typically decreases as we add more threads due to:

1. **Thread Management Overhead**: Creating and managing more threads consumes resources
2. **Synchronization Costs**: More threads require more synchronization
3. **Cache Contention**: Multiple threads competing for cache lines can cause performance degradation
4. **Diminishing Returns**: After a certain point, adding more threads provides less benefit relative to the overhead

For this ray tracing workload, we expect efficiency to drop off at higher thread counts due to these factors.

### Efficiency Plots: std::thread

### Efficiency Plots: OpenMP

## 5. Debug vs. Release Build Impact Analysis

### Overview

Release builds typically include compiler optimizations (like `-O3`) that can significantly improve performance. Let's analyze how much the Release build improves over Debug.

### Debug vs Release Performance Comparison Plots

## 6. Summary and Key Findings

### Key Observations:

#### Debug vs Release Build Impact
- Release builds show **significant performance improvements** over Debug builds
- The improvement factor is typically **2-5x** depending on ray count and thread configuration
- This is expected due to compiler optimizations (O3 level) enabled in Release mode
- The impact is consistent across different thread counts and ray counts

#### Speedup and Parallelization
- Both std::thread and OpenMP show good speedup with increased thread count
- Speedup generally improves with higher ray counts (more work per thread)
- Diminishing returns are observed at higher thread counts (8-16 threads)

#### Parallel Efficiency
- Efficiency drops off as thread count increases
- Lower ray counts show lower efficiency due to insufficient work per thread
- Higher ray counts maintain better efficiency at higher thread counts
- The ray tracing workload benefits from parallelization but with expected overhead

#### std::thread vs OpenMP
- Compare the relative performance and scaling characteristics
- Note any differences in efficiency curves
- Consider the implications for your implementation choice